
Notebook này được chỉnh lại theo hướng **6 bài / 6 điểm**, trong đó các bài whiteboard không hỏi lý thuyết suông mà yêu cầu sinh viên **chạy thuật toán bằng tay** hoặc **tinh chỉnh thuật toán cho phù hợp đề bài**.

| Bài | Hình thức | Nội dung chính | Điểm |
|---|---|---|---:|
| 1 | Whiteboard thuật toán | Chạy RL loop + tính return + chỉnh policy | 1 |
| 2 | Code | Value Iteration trên FrozenLake | 1 |
| 3 | Whiteboard thuật toán | Chạy Bellman backup / Q-learning backup bằng tay | 1 |
| 4 | Code | Tabular Q-learning | 1 |
| 5 | Code | DQN | 1 |
| 6 | Whiteboard tinh chỉnh + code ngắn | REINFORCE / Actor-Critic | 1 |

Giới hạn kiến thức bám sát slide ban đầu: MDP, policy, return, Bellman equation, Value Iteration, Q-learning, DQN, REINFORCE, Actor-Critic.



# Setup


In [ ]:

# Nếu chạy trên Colab và thiếu thư viện:
# !pip install gymnasium[classic-control,toy-text] matplotlib numpy torch


In [ ]:

import random
import numpy as np
import matplotlib.pyplot as plt
from collections import deque, namedtuple

import gymnasium as gym

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    import torch.nn.functional as F
except ImportError:
    torch = None
    print("Torch chưa được cài. Bài DQN / REINFORCE cần PyTorch.")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
if torch is not None:
    torch.manual_seed(SEED)

def reset_env(env):
    out = env.reset(seed=SEED)
    return out[0] if isinstance(out, tuple) else out

def step_env(env, action):
    out = env.step(action)
    if len(out) == 5:
        next_state, reward, terminated, truncated, info = out
        return next_state, reward, terminated or truncated, info
    return out



# Bài 1 — Whiteboard thuật toán: Chạy RL loop và tính return

Cho grid 3x3 sau:

```text
S  .  .
.  H  .
.  .  G
```

Ký hiệu:

- `S`: start state tại `(0,0)`
- `H`: hole tại `(1,1)`, rơi vào thì episode kết thúc, reward = `-1`
- `G`: goal tại `(2,2)`, đến nơi thì episode kết thúc, reward = `+1`
- Các ô thường reward = `0`
- Action: `UP`, `DOWN`, `LEFT`, `RIGHT`
- Nếu đi ra ngoài biên thì đứng yên, reward = `0`

Cho policy cố định:

```text
Từ mọi ô thường:
nếu có thể đi RIGHT thì đi RIGHT,
nếu không thì đi DOWN.
```

## Yêu cầu

1. Viết trajectory từ `S` theo policy trên.
2. Ghi rõ từng bước: `state -> action -> next_state -> reward`.
3. Tính return \(G_0\) với \(\gamma = 1\).
4. Tính return \(G_0\) với \(\gamma = 0.9\).
5. Policy này có tốt không? Nếu không, chỉnh policy bằng tay để đến goal mà tránh hole.


## Phần làm bài của sinh viên

Viết lời giải / tính toán của bạn ở đây.



# Bài 2 — Code: Value Iteration trên FrozenLake

Bài này kiểm tra phần **known MDP**.

Vì FrozenLake có transition model `P(s'|s,a)`, ta có thể dùng Bellman optimality equation:


$$
V^*(s) = \max_a \sum_{s'} P(s' \mid s,a)\Bigl[r(s,a,s') + \gamma V^*(s')\Bigr]
$$


## Yêu cầu

Hoàn thiện hàm `value_iteration`.

Sau đó in ra:

1. `V*`
2. greedy policy tương ứng


In [ ]:

def value_iteration(env, gamma=0.99, theta=1e-8, max_iter=10_000):
    nS = env.observation_space.n
    nA = env.action_space.n
    P = env.unwrapped.P

    V = np.zeros(nS)

    for it in range(max_iter):
        delta = 0
        new_V = V.copy()

        for s in range(nS):
            action_values = []

            for a in range(nA):
                q_sa = 0

                # TODO:
                # Duyệt các transition có thể xảy ra từ state s, action a
                # P[s][a] gồm các tuple: (prob, next_state, reward, done)
                for prob, next_s, reward, done in P[s][a]:
                    q_sa += prob * (reward + gamma * V[next_s] * (not done))

                action_values.append(q_sa)

            best_value = max(action_values)
            new_V[s] = best_value
            delta = max(delta, abs(V[s] - best_value))

        V = new_V

        if delta < theta:
            print(f"Converged after {it + 1} iterations")
            break

    policy = np.zeros(nS, dtype=int)

    for s in range(nS):
        action_values = []

        for a in range(nA):
            q_sa = 0
            for prob, next_s, reward, done in P[s][a]:
                q_sa += prob * (reward + gamma * V[next_s] * (not done))
            action_values.append(q_sa)

        policy[s] = int(np.argmax(action_values))

    return V, policy


env = gym.make("FrozenLake-v1", is_slippery=True)
V, policy = value_iteration(env)

print("V*:")
print(V.reshape(4, 4))

print("\nGreedy policy:")
print(policy.reshape(4, 4))

print("\nAction mapping: 0=LEFT, 1=DOWN, 2=RIGHT, 3=UP")
env.close()



# Bài 3 — Whiteboard thuật toán: Chạy Bellman backup và Q backup bằng tay

Cho MDP nhỏ có 3 state:

```text
s0: start
s1: intermediate
sT: terminal
```

Có 2 action tại `s0`:

```text
action A: đi đến s1, reward = 0
action B: đi đến sT, reward = 1
```

Tại `s1` chỉ có 1 action:

```text
action C: đi đến sT, reward = 3
```

`sT` là terminal, nên:

$$
V(s_T) = 0
$$

Cho \(\gamma = 0.9\).

## Yêu cầu A — Chạy Value backup

Giả sử ban đầu:

$$
V_0(s_0)=0,\quad V_0(s_1)=0,\quad V_0(s_T)=0
$$

Tính:

1. \(V_1(s_1)\)
2. \(V_1(s_0)\)
3. Action tốt nhất tại `s0` sau backup này là gì?

## Yêu cầu B — Chạy Q-learning backup

Giả sử hiện tại:

```text
Q(s0,A) = 0
Q(s0,B) = 0
Q(s1,C) = 0
```

Nếu agent đi transition:

```text
s0 --A, reward=0--> s1
```

Hãy cập nhật \(Q(s0,A)\) với:

```text
alpha = 0.5
gamma = 0.9
```

Dùng Q-learning update:

$$
Q(s,a) \leftarrow Q(s,a) + \alpha
\Bigl[r + \gamma \max_{a'} Q(s',a') - Q(s,a)\Bigr]
$$


## Phần làm bài của sinh viên

Viết lời giải / tính toán của bạn ở đây.



# Bài 4 — Code: Tabular Q-learning

Bài này kiểm tra phần **unknown MDP**.

Không dùng trực tiếp transition model nữa. Agent học bằng cách tương tác với environment.

Q-learning update:


$$
Q(s,a) \leftarrow Q(s,a) + \alpha \Bigl[r + \gamma \max_{a'} Q(s',a') - Q(s,a)\Bigr]
$$


## Yêu cầu

Hoàn thiện / chạy code Q-learning:

1. Train trên FrozenLake.
2. Vẽ moving average reward.
3. In learned policy.
4. Giải thích vai trò của epsilon-greedy.


In [ ]:

def epsilon_greedy(Q, state, epsilon, nA):
    if random.random() < epsilon:
        return random.randrange(nA)
    return int(np.argmax(Q[state]))

def train_q_learning(
    episodes=20_000,
    alpha=0.1,
    gamma=0.99,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay=0.9995,
    is_slippery=True
):
    env = gym.make("FrozenLake-v1", is_slippery=is_slippery)

    nS = env.observation_space.n
    nA = env.action_space.n
    Q = np.zeros((nS, nA))

    epsilon = epsilon_start
    episode_rewards = []

    for ep in range(episodes):
        state = reset_env(env)
        total_reward = 0

        for t in range(200):
            action = epsilon_greedy(Q, state, epsilon, nA)
            next_state, reward, done, info = step_env(env, action)

            # TODO: Q-learning target
            target = reward + gamma * np.max(Q[next_state]) * (not done)

            # TODO: TD error
            td_error = target - Q[state, action]

            # TODO: update Q
            Q[state, action] += alpha * td_error

            state = next_state
            total_reward += reward

            if done:
                break

        epsilon = max(epsilon_end, epsilon * epsilon_decay)
        episode_rewards.append(total_reward)

    env.close()
    return Q, episode_rewards

Q, rewards = train_q_learning()

window = 500
moving_avg = np.convolve(rewards, np.ones(window) / window, mode="valid")

plt.figure()
plt.plot(moving_avg)
plt.title("Q-learning on FrozenLake")
plt.xlabel("Episode")
plt.ylabel("Moving average reward")
plt.show()

print("Learned greedy policy:")
print(np.argmax(Q, axis=1).reshape(4, 4))
print("\nAction mapping: 0=LEFT, 1=DOWN, 2=RIGHT, 3=UP")



# Bài 5 — Code: Deep Q-learning / DQN trên CartPole

Bài này kiểm tra phần **function approximation**.

Tabular Q-learning không phù hợp khi state liên tục hoặc rất lớn.  
DQN thay Q-table bằng neural network:


$$
Q(s,a;w) \approx Q^*(s,a)
$$


Trong phạm vi slide, DQN cần:

1. Q-network
2. Epsilon-greedy
3. Experience replay
4. Fixed target network
5. MSE Bellman loss

## Yêu cầu

Chạy DQN trên CartPole và trả lời:

1. Replay buffer dùng để làm gì?
2. Target network dùng để làm gì?
3. Loss đến từ Bellman target như thế nào?


In [ ]:

if torch is not None:
    class QNetwork(nn.Module):
        def __init__(self, state_dim, action_dim):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(state_dim, 128),
                nn.ReLU(),
                nn.Linear(128, 128),
                nn.ReLU(),
                nn.Linear(128, action_dim)
            )

        def forward(self, x):
            return self.net(x)

    Transition = namedtuple("Transition", ["state", "action", "reward", "next_state", "done"])

    class ReplayBuffer:
        def __init__(self, capacity=50_000):
            self.buffer = deque(maxlen=capacity)

        def push(self, *args):
            self.buffer.append(Transition(*args))

        def sample(self, batch_size):
            batch = random.sample(self.buffer, batch_size)
            return Transition(*zip(*batch))

        def __len__(self):
            return len(self.buffer)

    def select_action(policy_net, state, epsilon, action_dim, device):
        if random.random() < epsilon:
            return random.randrange(action_dim)

        with torch.no_grad():
            state_t = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
            q_values = policy_net(state_t)
            return int(q_values.argmax(dim=1).item())


In [ ]:

def train_dqn_cartpole(
    episodes=150,
    batch_size=64,
    gamma=0.99,
    lr=1e-3,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay=0.995,
    target_update_every=10
):
    if torch is None:
        raise RuntimeError("Cần cài torch để chạy bài DQN.")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    env = gym.make("CartPole-v1")
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n

    policy_net = QNetwork(state_dim, action_dim).to(device)
    target_net = QNetwork(state_dim, action_dim).to(device)
    target_net.load_state_dict(policy_net.state_dict())
    target_net.eval()

    optimizer = optim.Adam(policy_net.parameters(), lr=lr)
    memory = ReplayBuffer()

    epsilon = epsilon_start
    returns = []

    for ep in range(episodes):
        state = reset_env(env)
        total_reward = 0

        for t in range(500):
            action = select_action(policy_net, state, epsilon, action_dim, device)
            next_state, reward, done, info = step_env(env, action)

            memory.push(state, action, reward, next_state, done)

            state = next_state
            total_reward += reward

            if len(memory) >= batch_size:
                batch = memory.sample(batch_size)

                states = torch.tensor(np.array(batch.state), dtype=torch.float32, device=device)
                actions = torch.tensor(batch.action, dtype=torch.long, device=device).unsqueeze(1)
                rewards = torch.tensor(batch.reward, dtype=torch.float32, device=device).unsqueeze(1)
                next_states = torch.tensor(np.array(batch.next_state), dtype=torch.float32, device=device)
                dones = torch.tensor(batch.done, dtype=torch.float32, device=device).unsqueeze(1)

                q_sa = policy_net(states).gather(1, actions)

                with torch.no_grad():
                    next_q = target_net(next_states).max(dim=1, keepdim=True)[0]
                    target = rewards + gamma * next_q * (1 - dones)

                loss = F.mse_loss(q_sa, target)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            if done:
                break

        epsilon = max(epsilon_end, epsilon * epsilon_decay)
        returns.append(total_reward)

        if (ep + 1) % target_update_every == 0:
            target_net.load_state_dict(policy_net.state_dict())

        if (ep + 1) % 25 == 0:
            print(f"Episode {ep+1}, average return={np.mean(returns[-25:]):.1f}, epsilon={epsilon:.3f}")

    env.close()
    return policy_net, returns

# Chạy nếu có torch:
# policy_net, dqn_returns = train_dqn_cartpole(episodes=150)
# plt.figure()
# plt.plot(dqn_returns)
# plt.title("DQN on CartPole")
# plt.xlabel("Episode")
# plt.ylabel("Return")
# plt.show()



# Bài 6 — Whiteboard tinh chỉnh thuật toán + REINFORCE code

Bài này kiểm tra phần **policy-based methods**.

Policy Gradient học trực tiếp:

$$
\pi_\theta(a \mid s)
$$

REINFORCE loss:

$$
L = -\sum_t \log \pi_\theta(a_t \mid s_t)G_t
$$

Actor-Critic thêm critic \(V(s)\), dùng advantage:

$$
A_t = G_t - V(s_t)
$$

## Phần A — Whiteboard: chạy policy-gradient update bằng tay

Cho một episode gồm 3 bước:

| t | action đã chọn | \(\log \pi(a_t \mid s_t)\) | discounted return \(G_t\) |
|---|---|---:|---:|
| 0 | RIGHT | -0.7 | 1.0 |
| 1 | RIGHT | -0.5 | 1.0 |
| 2 | DOWN | -1.2 | 1.0 |

REINFORCE loss:

$$
L = -\sum_t \log \pi(a_t \mid s_t)G_t
$$

Yêu cầu:

1. Tính loss.
2. Vì \(G_t > 0\), xác suất các action đã chọn nên tăng hay giảm?
3. Nếu episode có return âm, update sẽ đổi chiều như thế nào?

## Phần B — Tinh chỉnh thuật toán

Giả sử reward rất thưa: agent chỉ nhận `+1` ở cuối episode nếu thắng, còn lại toàn `0`.

Yêu cầu sinh viên đề xuất 2 chỉnh sửa trong phạm vi slide:

1. Có nên dùng discount return không? Vì sao?
2. Có nên thêm critic/baseline không? Vì sao?
3. Có nên giữ exploration không? Vì sao?


## Phần làm bài của sinh viên

Viết lời giải / tính toán của bạn ở đây.


In [ ]:

if torch is not None:
    class PolicyNetwork(nn.Module):
        def __init__(self, state_dim, action_dim):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(state_dim, 128),
                nn.ReLU(),
                nn.Linear(128, action_dim)
            )

        def forward(self, x):
            logits = self.net(x)
            return torch.distributions.Categorical(logits=logits)

    def compute_returns(rewards, gamma=0.99):
        returns = []
        G = 0

        for r in reversed(rewards):
            G = r + gamma * G
            returns.append(G)

        returns.reverse()
        return returns


In [ ]:

def train_reinforce_cartpole(episodes=300, gamma=0.99, lr=1e-2):
    if torch is None:
        raise RuntimeError("Cần cài torch để chạy REINFORCE.")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    env = gym.make("CartPole-v1")
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n

    policy = PolicyNetwork(state_dim, action_dim).to(device)
    optimizer = optim.Adam(policy.parameters(), lr=lr)

    episode_returns = []

    for ep in range(episodes):
        state = reset_env(env)

        log_probs = []
        rewards = []
        total_reward = 0

        for t in range(500):
            state_t = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
            dist = policy(state_t)
            action = dist.sample()

            log_probs.append(dist.log_prob(action))

            next_state, reward, done, info = step_env(env, int(action.item()))

            rewards.append(reward)
            total_reward += reward
            state = next_state

            if done:
                break

        returns = torch.tensor(compute_returns(rewards, gamma), dtype=torch.float32, device=device)

        # Normalize để giảm variance
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        loss = 0
        for log_prob, G in zip(log_probs, returns):
            loss += -log_prob * G

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        episode_returns.append(total_reward)

        if (ep + 1) % 50 == 0:
            print(f"Episode {ep+1}, average return={np.mean(episode_returns[-50:]):.1f}")

    env.close()
    return policy, episode_returns

# Chạy nếu có torch:
# pg_policy, pg_returns = train_reinforce_cartpole(episodes=300)
# plt.figure()
# plt.plot(pg_returns)
# plt.title("REINFORCE on CartPole")
# plt.xlabel("Episode")
# plt.ylabel("Return")
# plt.show()



# Rubric chấm 6 điểm

| Bài | Tiêu chí | Điểm |
|---|---|---:|
| 1 | Chạy đúng trajectory, tính đúng return, biết chỉnh policy tránh hole | 1 |
| 2 | Cài đúng Value Iteration và extract policy | 1 |
| 3 | Chạy đúng Bellman backup / Q backup, phân biệt synchronous và in-place update | 1 |
| 4 | Cài đúng Q-learning update + epsilon-greedy + đọc được policy | 1 |
| 5 | Chạy/giải thích đúng DQN, replay buffer, target network, Bellman loss | 1 |
| 6 | Tính được REINFORCE loss, giải thích chiều update, biết đề xuất critic/exploration cho reward thưa | 1 |

Tổng: **6 điểm**



# Checklist ôn nhanh trước khi nộp

Bạn nên làm được, không chỉ nói định nghĩa:

1. Mô phỏng được một trajectory `state -> action -> reward -> next_state`.
2. Tính được discounted return từ một reward sequence.
3. Chạy được một Bellman backup bằng tay.
4. Từ \(V(s)\), chọn được action bằng one-step lookahead.
5. Từ \(Q(s,a)\), chọn được action bằng `argmax`.
6. Cập nhật được một bước Q-learning bằng số cụ thể.
7. Giải thích được epsilon-greedy khi policy đang học sai.
8. Giải thích được replay buffer và target network trong DQN.
9. Tính được REINFORCE loss từ log-prob và return.
10. Biết khi nào nên thêm critic/baseline để giảm variance.
